In [1]:
import sys
from pathlib import Path

import json
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt

import geopandas as gpd
from shapely.geometry import Point

# UMAP + PCA
import umap
from sklearn.decomposition import PCA

# Ensure we can import repo modules (main.py, etc.)
REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [2]:
from main import Location2TextLightningModule

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
geoclip_ckpt_dir = "/home/libe2152/outputs/explainable-earth-embeddings/geoclip/pretrained/checkpoints"
geoclip_ckpt_path = Path(geoclip_ckpt_dir) / "location2text_pretrained.ckpt"

geoclip_model = Location2TextLightningModule.load_from_checkpoint(
    str(geoclip_ckpt_path),
    map_location=device,
    location_model_type="geoclip",
    location_model=None,
    location_model_filename=None,
    text_model_type="geoclip",
    text_model="geoclip",
    text_vocabulary="openai",
    finetune_mode="none",  # freeze all text model weights
    train_text_model=False
)
geoclip_model.to(device)

if geoclip_ckpt_path.exists():
    ckpt = torch.load(str(geoclip_ckpt_path), map_location=device, weights_only=False)
    state_dict = ckpt["state_dict"] if "state_dict" in ckpt else ckpt
    geoclip_model.load_state_dict(state_dict, strict=False)
else:
    print(f"Checkpoint not found at {geoclip_ckpt_path}. Using GeoCLIP init weights only.")

geoclip_model = geoclip_model.to(device).eval()
print("device:", device)
print("output_dim:", geoclip_model.output_dim)

train_text_model False


Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


[LocationEmbeddingModel] GeoCLIP (or non-SatCLIP) backend: using locations as [lat, lon] without reordering. Example[0]=[0.0, 0.0]


Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


device: cuda
output_dim: 512


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
satclip_ckpt_dir = "/home/libe2152/outputs/explainable-earth-embeddings/satclip/geoyfcc-text/checkpoints"
satclip_ckpt_path = Path(satclip_ckpt_dir) / "Text2Location-epoch=46-val_loss=1.3700.ckpt"  # adjust if needed

satclip_model = Location2TextLightningModule.load_from_checkpoint(
    str(satclip_ckpt_path),
    map_location=device,
)
satclip_model.to(device)

if satclip_ckpt_path.exists():
    ckpt = torch.load(str(satclip_ckpt_path), map_location=device, weights_only=False)
    state_dict = ckpt["state_dict"] if "state_dict" in ckpt else ckpt
    satclip_model.load_state_dict(state_dict, strict=False)
else:
    print(f"Checkpoint not found at {satclip_ckpt_path}. Using GeoCLIP init weights only.")

satclip_model = satclip_model.to(device).eval()
print("device:", device)
print("output_dim:", satclip_model.output_dim)

train_text_model True
using pretrained moco vit16
[LocationEmbeddingModel] SatCLIP backend: interpreting inputs as [lat, lon] and reordering to [lon, lat]. Example before[0]=[0.0, 0.0], after[0]=[0.0, 0.0]


Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


device: cuda
output_dim: 256


# Create concept embeddings with GeoCLIP and SatCLIP trained text encoders

In [7]:
import json
from pathlib import Path

In [6]:
@torch.no_grad()
def embed_texts(model, texts, device, batch_size=256, normalize=True):
    out = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding texts"):
        chunk = texts[i:i+batch_size]
        emb = model.text_model_predict(chunk, normalize=normalize)
        if emb.ndim == 1:
            emb = emb.unsqueeze(0)
        out.append(emb)
    return torch.cat(out, dim=0)

In [8]:
# Load geospatial concepts (list of strings)
vocab_dir = "/home/libe2152/projects/explainable-earth-embeddings/0_vocabs"  # adjust
geo_path = Path(vocab_dir) / "geoyfcc_concept_set.json"
with geo_path.open("r", encoding="utf-8") as f:
    geo_data = json.load(f)

if isinstance(geo_data, dict):
    geo_texts = list(geo_data.keys())
else:
    geo_texts = [str(x) for x in geo_data]

print(f"# geospatial concepts: {len(geo_texts)}")

# geospatial concepts: 10000


In [ ]:
# Embed them with the same model + preprocessing you used for desc_emb
geoclip_concept_emb = embed_texts(geoclip_model, geo_texts, device=device, batch_size=256, normalize=True).cpu()

print("geoclip_concept_emb:", geoclip_concept_emb.shape)

Embedding texts: 100%|██████████| 40/40 [00:01<00:00, 29.70it/s]

geo_emb: torch.Size([10000, 512])


In [ ]:
# Embed them with the same model + preprocessing you used for desc_emb
satclip_concept_emb = embed_texts(satclip_model, geo_texts, device=device, batch_size=256, normalize=True).cpu()

print("satclip_concept_emb:", satclip_concept_emb.shape)